## Loading the ontology

In [9]:
from pathlib import Path

from rdflib import OWL, RDF, RDFS, Graph, Namespace, URIRef

BASE_DIR = Path.cwd().parent
ONTOLOGY_PATH = BASE_DIR / "data" / "wittgenstein.owl"

if not ONTOLOGY_PATH.exists():
    raise FileNotFoundError(f"Ontology file not found: {ONTOLOGY_PATH}")

graph = Graph()
graph.parse(str(ONTOLOGY_PATH))

WAB = Namespace("http://purl.org/wittgensteinsource/ont/")
print(f"Loaded triples: {len(graph):,}")


Loaded triples: 749,032


## Inspecting the relevant class

In [10]:
part_class = WAB.Part

print("Class: ", part_class)
print("label: ", graph.value(part_class, RDFS.label))
print("comment: ", graph.value(part_class, RDFS.comment))

for parent in graph.objects(part_class, RDFS.subClassOf):
    print("SubClassOf: ", parent)

part_instances = list(graph.subjects(RDF.type, part_class))
print(f"number of Part instances: {len(part_instances):,}")

Class:  http://purl.org/wittgensteinsource/ont/Part
label:  Part
comment:  Part
SubClassOf:  http://purl.org/wittgensteinsource/ont/WittgensteinSource
number of Part instances: 7,881


## inspect a small sample of Part instances

In [11]:
def guess_work_from_label(label):
    if " (" in label:
        return label.split(" (")[0]
    return label


def get_label(thing):
    label = graph.value(thing, RDFS.label)
    return str(label) if label else ""


relation_names = [
    WAB.isPartOf,
    WAB.isWorkPublishedFrom,
    WAB.isPublishedInWork,
    WAB.hasPart,
]

seen_works = set()
shown = 0

for instance in part_instances:
    label = get_label(instance)
    if not label:
        continue

    work = guess_work_from_label(label)
    if work in seen_works:
        continue
    seen_works.add(work)

    print(f"\nLabel: {label}")
    print(f"URI: {instance}")

    for predicate in relation_names:
        short = predicate.split("/")[-1]
        for obj in graph.objects(instance, predicate):
            print(f"  outgoing: {short} -> {obj}")
        for subject in graph.subjects(predicate, instance):
            print(f"  incoming: {short} <- {subject}")

    
    shown += 1
    if shown >= 10:
        break

print(f"\nDistinct works shown: {shown}")


Label: Prototractatus (PT, eds. 1971 McGuinness, Nyberg et von Wright) 0
URI: http://purl.org/wittgensteinsource/ont/instances/W-PT%3A%200
  outgoing: isPartOf -> http://purl.org/wittgensteinsource/ont/instances/W-PT
  outgoing: isWorkPublishedFrom -> http://wittgensteinsource.org/Ms-104%2CIII%5B1%5D_d
  outgoing: isWorkPublishedFrom -> http://wittgensteinsource.org/Ms-104%2CV%5B1%5D_d
  outgoing: isWorkPublishedFrom -> http://wittgensteinsource.org/Ms-104%2CVI%5B1%5D_d
  outgoing: isWorkPublishedFrom -> http://wittgensteinsource.org/Ms-104%2C1%5B1%5D_d
  outgoing: isWorkPublishedFrom -> http://wittgensteinsource.org/Ms-104%2C119%5B1%5Det120%5B1%5Det121%5B1%5D_d
  incoming: isPublishedInWork <- http://wittgensteinsource.org/Ms-104%2CIII%5B1%5D_d
  incoming: isPublishedInWork <- http://wittgensteinsource.org/Ms-104%2CV%5B1%5D_d
  incoming: isPublishedInWork <- http://wittgensteinsource.org/Ms-104%2CVI%5B1%5D_d
  incoming: isPublishedInWork <- http://wittgensteinsource.org/Ms-104%2C1%5B

## Inspecting proposition 6.23

In [12]:
from rdflib import Literal

target_label = "Tractatus logico-philosophicus (TLP, 1922 et 1933) 6.23"
target_matches = list(graph.subjects(RDFS.label, Literal(target_label)))

print("Matches:", len(target_matches))
for target in target_matches:
    print("WAB individual:", target)
    print("  type:", graph.value(target, RDF.type))
    print("  label:", graph.value(target, RDFS.label))
    for parent in graph.objects(target, WAB.isPartOf):
        print("  isPartOf ->", parent)
    for source in graph.objects(target, WAB.isWorkPublishedFrom):
        print("  isWorkPublishedFrom ->", source)
    for container in graph.subjects(WAB.hasPart, target):
        print("  incoming hasPart <-", container)

Matches: 1
WAB individual: http://purl.org/wittgensteinsource/ont/instances/W-TLP%3A%206.23
  type: http://purl.org/wittgensteinsource/ont/Part
  label: Tractatus logico-philosophicus (TLP, 1922 et 1933) 6.23
  isPartOf -> http://purl.org/wittgensteinsource/ont/instances/W-TLP
  isWorkPublishedFrom -> http://wittgensteinsource.org/Ts-202%2C46r%5B8%5D_d
  isWorkPublishedFrom -> http://wittgensteinsource.org/Ts-203%2C50r%5B3%5D_d
  isWorkPublishedFrom -> http://wittgensteinsource.org/Ts-204%2C46r%5B8%5D_d
  incoming hasPart <- http://purl.org/wittgensteinsource/ont/instances/W-TLP


## Checking language and text sources

In [13]:
from rdflib import Literal

english_literals = []

for subject, predicate, obj in graph:
    if isinstance(obj, Literal) and obj.language:  # noqa: SIM102
        if obj.language.lower().startswith("en"):
            english_literals.append(
                (subject, predicate, obj)
            )

print("English-tagged literals:", len(english_literals))

for subject, predicate, obj in english_literals[:30]:
    print("\nSubject:", subject)
    print("Property:", predicate)
    print("Text:", obj)

English-tagged literals: 1

Subject: http://purl.org/wittgensteinsource/ont/hasVariant
Property: http://www.w3.org/2000/01/rdf-schema#comment
Text: A property for stating that a Bemerkung is or contains a variant of another 
Bemerkung.


In [14]:
from rdflib import URIRef

target = URIRef(
    "http://purl.org/wittgensteinsource/ont/instances/W-TLP%3A%206.23"
)

print("outgoing triples:")
for s, p, o in graph.triples((target, None, None)):
    print(f"{p.split("/")[-1]} -> {o}")

print("incoming triples:")
for s, p, o in graph.triples((None, None, target)):
    print(f"{s} -{p.split("/")[-1]}->target")

outgoing triples:
22-rdf-syntax-ns#type -> http://purl.org/wittgensteinsource/ont/Part
rdf-schema#label -> Tractatus logico-philosophicus (TLP, 1922 et 1933) 6.23
isPartOf -> http://purl.org/wittgensteinsource/ont/instances/W-TLP
isWorkPublishedFrom -> http://wittgensteinsource.org/Ts-202%2C46r%5B8%5D_d
isWorkPublishedFrom -> http://wittgensteinsource.org/Ts-203%2C50r%5B3%5D_d
isWorkPublishedFrom -> http://wittgensteinsource.org/Ts-204%2C46r%5B8%5D_d
incoming triples:
http://purl.org/wittgensteinsource/ont/instances/W-TLP -hasPart->target
http://wittgensteinsource.org/Ts-202%2C46r%5B8%5D_d -isPublishedInWork->target
http://wittgensteinsource.org/Ts-203%2C50r%5B3%5D_d -isPublishedInWork->target
http://wittgensteinsource.org/Ts-204%2C46r%5B8%5D_d -isPublishedInWork->target


In [15]:
from rdflib import RDF, RDFS, Namespace

print("Statements about the class Part:")
for triple in graph.triples((part_class, None, None)):
    print(triple)

print("\nStatements referring to Part (first 20):")
shown = 0
for triple in graph.triples((None, None, part_class)):
    print(triple)
    shown += 1
    if shown >= 20:
        break

print(f"\n... and {len(part_instances) - shown:,} more")
print(f"Total Part instances: {len(part_instances):,}")

Statements about the class Part:
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#Class'))
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#comment'), rdflib.term.Literal('Part', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/WittgensteinSource'))
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label'), rdflib.term.Literal('Part', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))

Statements referring to Part (first 20):
(rdflib.term.BNode('N4221ce4

In [17]:
print("Statements about the class Part:")
for triple in graph.triples((part_class, None, None)):
    print(triple)

print("\nStatements referring to Part (first 20):")
shown = 0
for triple in graph.triples((None, None, part_class)):
    print(triple)
    shown += 1
    if shown >= 20:
        break

print(f"\n... and {len(part_instances) - shown:,} more")
print(f"Total Part instances: {len(part_instances):,}")

Statements about the class Part:
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#Class'))
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#comment'), rdflib.term.Literal('Part', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/WittgensteinSource'))
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label'), rdflib.term.Literal('Part', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))

Statements referring to Part (first 20):
(rdflib.term.BNode('N4221ce4

exploring ontology classes

In [18]:
declared_classes = set()
for c in graph.subjects(RDF.type, OWL.Class):
    declared_classes.add(c)
for c in graph.subjects(RDF.type, RDFS.Class):
    declared_classes.add(c)

counts = {}
for subject, class_uri in graph.subject_objects(RDF.type):
    if class_uri in declared_classes:
        if class_uri in counts:
            counts[class_uri] += 1
        else:
            counts[class_uri] = 1

sorted_items = sorted(counts.items(), key=lambda pair: pair[1], reverse=True)

print("Number of classes with instances:", len(counts))
for class_uri, count in sorted_items:
    label = graph.value(class_uri, RDFS.label)
    print(f"{label or class_uri}: {count}")

Number of classes with instances: 10
Nachlass Bemerkung: 54626
MS: 32189
TS: 22600
Part: 7881
Date: 2206
WittgensteinExternalSource: 428
Person: 343
Work: 71
Text subgenre: 13
Language: 4


In [19]:
for class_uri, count in sorted_items:
    label = graph.value(class_uri, RDFS.label) or class_uri
    comment = graph.value(class_uri, RDFS.comment) or "(no comment)"

    print("\nCLASS:", label)
    print("URI:", class_uri)
    print("COUNT:", count)
    print("COMMENT:", comment)


CLASS: Nachlass Bemerkung
URI: http://purl.org/wittgensteinsource/ont/Bemerkung
COUNT: 54626
COMMENT: Bemerkung, subclass of Wittgenstein Primary source: Corresponding roughly to a
            single remark in a Wittgenstein primary source, identified by the
            Wittgenstein Archives through a “siglum” which is composed of item identifier, page
            identifier(s) and position identifier(s). Example of instance: Ts-310,21[2]et22[1]
        

CLASS: MS
URI: http://purl.org/wittgensteinsource/ont/MS
COUNT: 32189
COMMENT: MS, subclass of Wittgenstein Primary Source: A Wittgenstein primary source, handwritten. Example of instance: Ms-115; Ms-139a 

CLASS: TS
URI: http://purl.org/wittgensteinsource/ont/TS
COUNT: 22600
COMMENT: TS, subclass of Wittgenstein Primary source: A Wittgenstein primary source, typed. Example of instance: Ts-213; Ts-310 

CLASS: Part
URI: http://purl.org/wittgensteinsource/ont/Part
COUNT: 7881
COMMENT: Part

CLASS: Date
URI: http://purl.org/wittgenstei